# 并发专题

`stdlib.md` 里的 `threading`、`asyncio`、`concurrent.futures` 讲的是每个模块怎么用。这个专题只回答一个问题：**同一份活，该用线程、进程，还是协程。**

结论先放这里，题自己量：

| 瓶颈 | 用什么 | 原因 |
| --- | --- | --- |
| 烧 CPU | 进程，或 3.14 的子解释器 | GIL 让线程一次只有一个在跑 Python 字节码 |
| 等 I/O | 线程或协程都可以 | 等待时 GIL 是放开的，多线程能真并发 |

笔记本在 `codes/concurrency/qa.ipynb`，负载在旁边的 `bench.py`。

先运行下一格。每题的 `# 作答` 格里已经写好了能跑的答案，直接从上往下跑一遍就能看到结论。想自己练，就先清空那一格再写。

In [1]:
import asyncio
import os
import sys
import time
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from pathlib import Path


def lab_root() -> Path:
    """qa.ipynb 所在目录。"""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "concurrency" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "concurrency"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    return here


def timed(fn):
    """跑 fn，返回 (耗时秒数, 返回值)。只用于同步函数。"""
    start = time.perf_counter()
    value = fn()
    return time.perf_counter() - start, value


async def timed_async(make_coro):
    """跑一个协程，返回 (耗时秒数, 返回值)。笔记本里用它，别用 asyncio.run。"""
    start = time.perf_counter()
    value = await make_coro()
    return time.perf_counter() - start, value


ROOT = lab_root()
# 进程池的子进程要能 import bench，路径得进 sys.path。
sys.path.insert(0, str(ROOT))

import bench

print("ROOT:", ROOT)
print("CPU 核数:", os.cpu_count())
print("一次 cpu_work:", round(timed(bench.cpu_work)[0], 3), "s")

ROOT: /Users/keyficller/Documents/AEFS-Notes/codes/concurrency
CPU 核数: 10
一次 cpu_work: 0.126 s


## 1. GIL：线程为什么不能加速 CPU

同一份 `bench.cpu_work` 跑 4 次，量三种写法：

1. 串行：`[bench.cpu_work() for _ in range(4)]`
2. 线程池：`ThreadPoolExecutor(4)`
3. 进程池：`ProcessPoolExecutor(4)`

再把第四个也量上：3.14 新增的 `InterpreterPoolExecutor(4)`（子解释器，各有各的 GIL）。

回答：线程比串行快多少？进程比串行快多少？为什么进程没快到 4 倍？

In [2]:
# 作答：串行、线程池、进程池、子解释器，各量一次

serial, _ = timed(lambda: [bench.cpu_work() for _ in range(4)])

with ThreadPoolExecutor(4) as ex:
    threads, _ = timed(lambda: list(ex.map(bench.cpu_work, [bench.N] * 4)))

with ProcessPoolExecutor(4) as ex:
    procs, _ = timed(lambda: list(ex.map(bench.cpu_work, [bench.N] * 4)))

from concurrent.futures import InterpreterPoolExecutor

with InterpreterPoolExecutor(4) as ex:
    interps, _ = timed(lambda: list(ex.map(bench.cpu_work, [bench.N] * 4)))

for name, t in [("串行", serial), ("线程", threads), ("进程", procs), ("子解释器", interps)]:
    print(f"{name:6} {t:5.2f}s")

串行      0.46s
线程      0.46s
进程      0.22s
子解释器    0.16s


## 2. I/O：同样的写法，结论反过来

`bench.io_work(0.2)` 只是 `time.sleep`，不烧 CPU。跑 8 次，量三种写法：

1. 串行
2. `ThreadPoolExecutor(8)`
3. `asyncio.gather` + `asyncio.to_thread`

笔记本里事件循环已经在跑，异步的格子直接用顶层 `await`，计时用前置格里给的 `timed_async`。为什么不能 `asyncio.run` 见第 7 题。

回答：这一题里线程和协程都快了吗？为什么同一套线程，上一题没用、这一题有用？

In [3]:
# 作答：串行、线程池、asyncio.gather，各量一次

serial_io, _ = timed(lambda: [bench.io_work(0.2) for _ in range(8)])

with ThreadPoolExecutor(8) as ex:
    threads_io, _ = timed(lambda: list(ex.map(bench.io_work, [0.2] * 8)))


async def gather_io():
    return await asyncio.gather(*[asyncio.to_thread(bench.io_work, 0.2) for _ in range(8)])


aio_io, _ = await timed_async(gather_io)

for name, t in [("串行", serial_io), ("线程", threads_io), ("协程", aio_io)]:
    print(f"{name:6} {t:5.2f}s")

串行      1.63s
线程      0.21s
协程      0.20s


## 3. 限流：一次只放 3 个

调模型接口时不能一次打满。发 9 个请求，每个等 0.2 秒，用 `asyncio.Semaphore(3)` 限制最多 3 个同时跑，量总耗时。

顺便在临界区里记一下实际同时跑的最大数量，验证信号量真的限住了。

回答：总耗时接近几批？如果把上限改成 9，要多久？

In [4]:
# 作答：9 个请求，上限 3 个

async def limited(n: int, cap: int) -> int:
    sem = asyncio.Semaphore(cap)
    live = 0
    peak = 0

    async def one(i: int) -> None:
        nonlocal live, peak
        async with sem:
            live += 1
            peak = max(peak, live)
            await asyncio.sleep(0.2)
            live -= 1

    await asyncio.gather(*[one(i) for i in range(n)])
    return peak


elapsed, peak = await timed_async(lambda: limited(9, 3))
print(f"耗时 {elapsed:.2f}s，峰值并发 {peak}")

耗时 0.60s，峰值并发 3


## 4. 谁先回来先处理

三个任务分别等 `0.3`、`0.1`、`0.2` 秒。用 `asyncio.as_completed` 按完成顺序打印名字。

再用 `asyncio.gather` 做一遍，看顺序有什么不同。

回答：什么时候必须用 `as_completed`？流式输出该用哪个？

In [5]:
# 作答：先 as_completed，再 gather，对比顺序

async def finish(name: str, delay: float) -> str:
    await asyncio.sleep(delay)
    return name


async def by_done():
    tasks = [finish("a", 0.3), finish("b", 0.1), finish("c", 0.2)]
    return [await t for t in asyncio.as_completed(tasks)]


async def by_input():
    return await asyncio.gather(finish("a", 0.3), finish("b", 0.1), finish("c", 0.2))


print("as_completed:", await by_done())
print("gather      :", await by_input())

as_completed: ['b', 'c', 'a']


gather      : ['a', 'b', 'c']


## 5. 一个失败，别拖垮整批

5 个请求，第 3 个抛 `RuntimeError`。

1. 先直接 `asyncio.gather(...)`，看它抛出什么。
2. 加 `return_exceptions=True`，把成功和异常都收回来。
3. 再加一个：用 `asyncio.wait_for` 等一个要 1 秒的任务，`timeout=0.2`。

回答：`gather` 默认失败后其余任务还会跑完吗？「部分成功」的报告怎么拼？

In [6]:
# 作答：默认行为、return_exceptions、wait_for 超时

async def boom(i: int) -> int:
    if i == 2:
        raise RuntimeError(f"req {i} failed")
    return i


async def strict():
    return await asyncio.gather(*[boom(i) for i in range(5)])


async def loose():
    return await asyncio.gather(*[boom(i) for i in range(5)], return_exceptions=True)


async def too_slow():
    try:
        await asyncio.wait_for(asyncio.sleep(1), timeout=0.2)
    except TimeoutError:
        return "超时放弃"


try:
    await strict()
except RuntimeError as exc:
    print("默认:", type(exc).__name__, exc)

print("收住:", await loose())
print("超时:", await too_slow())

默认: RuntimeError req 2 failed
收住: [0, 1, RuntimeError('req 2 failed'), 3, 4]


超时: 超时放弃


## 6. 一个阻塞函数，毁掉整条并发

同样 5 个协程，每个「等 0.2 秒」，两种写法：

1. 协程体里调 `time.sleep(0.2)`
2. 协程体里 `await asyncio.sleep(0.2)`

各量一次。

回答：为什么第一种没有并发？把 `time.sleep` 换成 `requests.get(...)` 会怎样？怎么改才对？

In [7]:
# 作答：两种写法各量一次

async def blocking():
    async def one():
        time.sleep(0.2)

    await asyncio.gather(*[one() for _ in range(5)])


async def yielding():
    async def one():
        await asyncio.sleep(0.2)

    await asyncio.gather(*[one() for _ in range(5)])


bad, _ = await timed_async(blocking)
good, _ = await timed_async(yielding)
print(f"time.sleep   {bad:.2f}s")
print(f"asyncio.sleep {good:.2f}s")

time.sleep   1.02s
asyncio.sleep 0.20s


## 7. 笔记本里的两个坑

**(a)** 在单元格里直接写 `asyncio.run(...)`。这一格**会报错**，报错本身就是答案。

**(b)** 在单元格里定义 `def work(n): ...`，交给 `ProcessPoolExecutor`。这一格也会失败。池子抛的是 `BrokenProcessPool`，真正的原因（`AttributeError: module '__main__' has no attribute 'work'`）在子进程的 stderr 里，往上翻第一条 Traceback。

然后再把 `bench.cpu_work` 交进去，能跑。

回答：

1. (a) 的报错说明了什么？笔记本里正确写法是什么？
2. (b) 为什么子进程里没有 `work`？为什么改用 `bench.cpu_work` 就好了？
3. 这三条约束里，哪些是笔记本特有的，哪些换成脚本也一样？

In [8]:
# 作答：(a) asyncio.run、(b) 单元格里定义的函数、再换成 bench 里的函数

# (a) 这一格故意报错
coro = asyncio.sleep(0)
try:
    asyncio.run(coro)
except RuntimeError as exc:
    print("(a)", type(exc).__name__, exc)
finally:
    coro.close()  # 没 await 的协程要关掉，不然会多一条 RuntimeWarning


# (b) 笔记本里定义的函数，进程池拿不到
def work(n: int) -> int:
    return n * 2


try:
    with ProcessPoolExecutor(2) as ex:
        print(list(ex.map(work, [1, 2])))
except Exception as exc:
    print("(b)", type(exc).__name__, exc)


# 换成模块里的函数就能跑
with ProcessPoolExecutor(2) as ex:
    print("(c)", list(ex.map(bench.cpu_work, [10, 20])))

(a) RuntimeError asyncio.run() cannot be called from a running event loop


(b) BrokenProcessPool A process in the process pool was terminated abruptly while the future was running or pending.


Process SpawnProcess-6:
Process SpawnProcess-5:
Traceback (most recent call last):
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.14/3.14.6/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/process.py", line 320, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/opt/homebrew/Cellar/python@3.14/3.14.6/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/process.py", line 320, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/opt/homebrew/Cellar/python@3.14/3.14.6/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/python@3.14/3.14.6/Frameworks/Python.framework/Versions/3.14/lib/python3.14/concurrent/futures/process.py", line 242, in _process_worker
    call_item = call_queue.get(block=True)
  File "/opt/homebrew/Cellar/python@3.14/3.14.6/Framew

(c) [364560, 864858]


## 跑完自检

`python check.py` 会把上面几题的结论逐条量一遍并断言：线程不加速 CPU、进程加速、线程加速 I/O、信号量真的限流、同步阻塞会毁掉并发。约 8 秒。

耗时断言的区间给得很宽，机器忙也不会误报；它防的是结论写反，不是测你的机器有多快。